# Recommender Systems — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/recsys/recsys-lab.ipynb)

Companion notebook for the **Recommender Systems** track (`rs-m1` … `rs-m12`,
`rs-b1` … `rs-b6`). It runs on **NovaCart** — the same 6-shopper, 9-product,
40-event store the site's widgets use — so every number here is the number in
the module text.

The store is deliberately tiny. You can check any result by hand, and that is
the point: a recommender you cannot hand-check is a recommender you cannot debug.

CPU only, no downloads, runs in seconds.

| Section | Modules | What runs |
|---|---|---|
| 1 | rs-m2, m3, b1 | the event frame, implicit weights, temporal split, the audit |
| 2 | rs-m4, b2 | Precision/Recall/MRR/MAP/NDCG, hand-checked |
| 3 | rs-m5 | temporal vs random vs leave-one-out, and the leaks each allows |
| 4 | rs-m6, b3 | random / popularity / recency / co-visitation baselines |
| 5 | rs-m7 | item-item and user-user collaborative filtering |
| 6 | rs-m8, b4 | implicit ALS with the Gram trick, monotone loss, k sweep |
| 7 | rs-m10, b5 | point-in-time features, `merge_asof`, the leak assertion |
| 8 | rs-m9, b6 | learning to rank with sampled negatives |
| 9 | rs-m11, m12 | negative sampling strategies, cold start |

## Setup — the NovaCart store

Nine products, six shoppers, forty implicit events over thirty days.
**P9 has zero events on purpose** — it is the cold-start item every model in
this notebook fails on until section 9.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

np.set_printoptions(precision=3, suppress=True, linewidth=120)
pd.set_option('display.width', 120)
TRACK = '#65a30d'

PRODUCTS = pd.DataFrame([
    ('P1', 'Wireless Earbuds', 'audio',   2499),
    ('P2', 'Phone Case',       'mobile',   399),
    ('P3', 'Laptop Stand',     'desk',    1299),
    ('P4', 'Mech. Keyboard',   'desk',    4999),
    ('P5', 'USB-C Hub',        'desk',    1899),
    ('P6', 'Coffee Grinder',   'kitchen', 3499),
    ('P7', 'Yoga Mat',         'fitness',  899),
    ('P8', 'Steel Bottle',     'fitness',  649),
    ('P9', 'ANC Headphones',   'audio',   7999),   # new arrival: zero events
], columns=['item_id', 'name', 'cat', 'price'])

USERS = pd.DataFrame([('U1','Aarav'),('U2','Bhavna'),('U3','Chen'),
                      ('U4','Diya'),('U5','Eli'),('U6','Farida')],
                     columns=['user_id', 'name'])

# (user, item, type, day) — day is 1..30
RAW = [
    ('U1','P2','view',2),('U1','P1','view',3),('U1','P4','view',5),('U1','P1','cart',6),
    ('U1','P5','view',9),('U1','P5','cart',11),('U1','P1','purchase',26),
    ('U2','P8','view',1),('U2','P1','view',3),('U2','P3','view',4),('U2','P4','view',7),
    ('U2','P4','cart',8),('U2','P5','view',12),('U2','P8','cart',14),('U2','P4','purchase',27),
    ('U3','P2','view',2),('U3','P5','view',5),('U3','P3','view',6),('U3','P3','cart',9),
    ('U3','P3','purchase',10),('U3','P8','view',13),('U3','P8','purchase',28),
    ('U4','P8','view',4),('U4','P7','view',6),('U4','P7','cart',8),('U4','P6','view',10),
    ('U4','P8','cart',15),('U4','P7','purchase',25),
    ('U5','P8','view',3),('U5','P1','view',5),('U5','P2','view',7),('U5','P1','cart',9),
    ('U5','P6','view',12),('U5','P6','cart',16),('U5','P5','purchase',29),
    ('U6','P8','view',2),('U6','P7','view',6),('U6','P3','view',11),('U6','P7','cart',13),
    ('U6','P1','purchase',30),
]

# rs-m2: one scalar per event type. A purchase is not "10 views" — it is a
# different KIND of evidence — but a single weight is the standard first move.
EVENT_WEIGHT = {'view': 1, 'cart': 3, 'purchase': 10}

events = pd.DataFrame(RAW, columns=['user_id', 'item_id', 'event_type', 'day'])
events['weight'] = events.event_type.map(EVENT_WEIGHT)
events['event_ts'] = pd.Timestamp('2026-02-28') + pd.to_timedelta(events.day, unit='D')

ITEMS = list(PRODUCTS.item_id)          # the FULL catalog, not just interacted items
USER_IDS = list(USERS.user_id)
name = dict(zip(PRODUCTS.item_id, PRODUCTS.name))

print(events.head(8).to_string(index=False))
print(f'\n{len(events)} events, {len(USER_IDS)} users, {len(ITEMS)} products')
print('events per type:', events.event_type.value_counts().to_dict())
assert len(events) == 40 and len(ITEMS) == 9

---
## 1 · rs-b1 · The split, and the audit that makes it trustworthy

The split must be **temporal**: everything the model sees happened before
everything it is scored on. The assertion below is the one line that keeps that
true after someone refactors the loader.

In [ ]:
CUTOFF = 24            # days 1..24 = train, 25..30 = the held-out future

def temporal_split(ev, cutoff):
    return ev[ev.day <= cutoff].copy(), ev[ev.day > cutoff].copy()

train, test = temporal_split(events, CUTOFF)

# The assertion that makes the split trustworthy. Run it every time.
assert train.day.max() < test.day.min(), 'temporal split leaks'

targets = (test.sort_values('day').groupby('user_id')['item_id'].last().to_dict())
print('held-out target per user:', targets)
assert targets == {'U1':'P1','U2':'P4','U3':'P8','U4':'P7','U5':'P5','U6':'P1'}

In [ ]:
def audit(train, test, targets, items):
    pairs = train.drop_duplicates(['user_id', 'item_id'])
    density = len(pairs) / (train.user_id.nunique() * len(items))
    print(f'train {len(train):>6}  test {len(test):>6}')
    print(f'users train {train.user_id.nunique()}  test {test.user_id.nunique()}')
    print(f'catalog {len(items)}  cold items {sorted(set(items) - set(train.item_id))}')
    print(f'cold users {len(set(test.user_id) - set(train.user_id))}')
    print(f'density {density:.4f}')
    assert train.day.max() < test.day.min()
    assert set(targets) <= set(test.user_id)
    assert not train.duplicated(['user_id','item_id','event_type','day']).any()
    return density

density = audit(train, test, targets, ITEMS)
assert len(train) == 34 and len(test) == 6
assert abs(density - 0.4259) < 1e-4
assert sorted(set(ITEMS) - set(train.item_id)) == ['P9']
print('\nP9 has zero training interactions. Every collaborative model below')
print('will score it 0 — that is cold start, and no amount of tuning fixes it.')

In [ ]:
# rs-m2: implicit feedback splits into PREFERENCE and CONFIDENCE.
#   r_ui = summed event weight,  p_ui = 1[r>0],  c_ui = 1 + alpha*r
R = (train.pivot_table(index='user_id', columns='item_id', values='weight',
                       aggfunc='sum', fill_value=0)
          .reindex(index=USER_IDS, columns=ITEMS, fill_value=0))
R = R.astype(float)
print('R — implicit strength (summed event weights)\n')
print(R.to_string())

alpha = 1.0
P = (R > 0).astype(float)
Cc = 1 + alpha * R
print(f'\nChen viewed P3 (1) + carted it (3) + bought it (10) -> r = {R.loc["U3","P3"]:.0f}, '
      f'p = {P.loc["U3","P3"]:.0f}, c = {Cc.loc["U3","P3"]:.0f}')
assert R.loc['U3','P3'] == 14 and Cc.loc['U3','P3'] == 15
print('The zeros are NOT negatives — they are unknowns carrying confidence 1.')

---
## 2 · rs-m4 / rs-b2 · The metric harness

Write the metrics, then **test them against values you worked out by hand**.
An untested metric will happily tell you a broken model is excellent.

**Reading the next cell.** Six metrics, and the differences are entirely about
*what each one refuses to notice*:

| metric | question | blind to |
|---|---|---|
| `precision@k` | of the k slots I spent, how many paid off | anything below k |
| `recall@k` | of everything they wanted, how much did I surface | *where* in the top-k it landed |
| `hit@k` | did I get **anything** right | how many, and where |
| `MRR` | how far down was the **first** hit | every hit after the first |
| `MAP` | precision measured again at each hit | position beyond the ordering |
| `NDCG` | log-discounted gain vs the best possible order | nothing much — which is why it is the default |

The `1/log2(i+2)` in `dcg_at_k` is the position discount: rank 1 is worth 1.0,
rank 2 is worth 0.63, rank 5 is worth 0.39. `+2` rather than `+1` because `i`
is zero-based and `log2(1) = 0` would divide by zero.

The tests underneath use a fact worth remembering: with **exactly one** relevant
item at rank `r`, `MRR = MAP = 1/r` and `NDCG = 1/log2(r+1)`. Closed forms make
the metric testable, and an untested metric will cheerfully rank a broken model
first.

In [ ]:
def precision_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & set(relevant)) / k

def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & set(relevant)) / len(relevant) if relevant else 0.0

def hit_at_k(ranked, relevant, k):
    return float(bool(set(ranked[:k]) & set(relevant)))

def reciprocal_rank(ranked, relevant, k=None):
    for i, it in enumerate(ranked[:k] if k else ranked):
        if it in relevant: return 1.0 / (i + 1)
    return 0.0

def average_precision_at_k(ranked, relevant, k):
    if not relevant: return 0.0
    hits = s = 0
    for i, it in enumerate(ranked[:k]):
        if it in relevant:
            hits += 1; s += hits / (i + 1)
    return s / min(k, len(relevant))

def dcg_at_k(ranked, relevant, k):
    return sum(1/np.log2(i+2) for i, it in enumerate(ranked[:k]) if it in relevant)

def ndcg_at_k(ranked, relevant, k):
    ideal = sum(1/np.log2(i+2) for i in range(min(k, len(relevant))))
    return dcg_at_k(ranked, relevant, k)/ideal if ideal else 0.0

# --- the tests. One relevant item at rank r has closed forms, so check them. ---
slate = ['P8','P1','P2','P5','P3','P7','P4','P6']
for r, item in [(1,'P8'), (2,'P1'), (4,'P5')]:
    rel = {item}
    assert abs(precision_at_k(slate, rel, 3) - (1/3 if r <= 3 else 0)) < 1e-12
    assert abs(reciprocal_rank(slate, rel) - 1/r) < 1e-12
    assert abs(average_precision_at_k(slate, rel, 8) - 1/r) < 1e-12
    assert abs(ndcg_at_k(slate, rel, 8) - 1/np.log2(r+1)) < 1e-12
print('closed-form checks pass for a single relevant item at ranks 1, 2 and 4')

rel3 = {'P1','P5','P7'}
print(f'\nslate {slate}, relevant {sorted(rel3)}')
for k in (1, 3, 5, 8):
    print(f'  k={k}  P {precision_at_k(slate,rel3,k):.3f}  R {recall_at_k(slate,rel3,k):.3f}  '
          f'MAP {average_precision_at_k(slate,rel3,k):.3f}  NDCG {ndcg_at_k(slate,rel3,k):.3f}')
print('\nRecall only rises, precision mostly falls, MRR stops moving after the first hit.')
print('NDCG is the only one that reacts to WHERE each later hit lands.')

In [ ]:
def catalog_coverage(ranked_per_user, k):
    shown = set()
    for r in ranked_per_user.values(): shown.update(r[:k])
    return len(shown) / len(ITEMS)

def evaluate(rank_fn, targets, k=3):
    rows, per = [], {}
    for u, t in targets.items():
        ranked = rank_fn(u)
        per[u] = ranked
        pos = ranked.index(t) + 1 if t in ranked else None
        rows.append(dict(user=u, target=t, rank=pos,
                         hit=hit_at_k(ranked, {t}, k),
                         rr=reciprocal_rank(ranked, {t}, k),
                         ndcg=ndcg_at_k(ranked, {t}, k),
                         ap=average_precision_at_k(ranked, {t}, k)))
    df = pd.DataFrame(rows)
    return dict(HitRate=df.hit.mean(), MRR=df.rr.mean(), NDCG=df.ndcg.mean(),
                MAP=df.ap.mean(), coverage=catalog_coverage(per, k)), df

print('The harness takes any rank_fn(user) -> list of item ids. Everything below')
print('plugs into it unchanged, which is what makes the comparison fair.')

---
## 3 · rs-m5 · Splits and the leaks they allow

A random split on a time-ordered log lets the model read the future. Here is
that leak, measured rather than asserted.

In [ ]:
rng = np.random.default_rng(11)
mask = rng.random(len(events)) >= 0.2
rand_train, rand_test = events[mask], events[~mask]

print(f'{"split":<16}{"train":>7}{"test":>6}   max train day vs min test day')
print(f'{"temporal":<16}{len(train):>7}{len(test):>6}   {train.day.max()} < {test.day.min()}   OK')
print(f'{"random":<16}{len(rand_train):>7}{len(rand_test):>6}   '
      f'{rand_train.day.max()} vs {rand_test.day.min()}   LEAK')
assert rand_train.day.max() > rand_test.day.min(), 'random split should leak here'

# How much future does the random split expose?
leaked = (rand_train.day > rand_test.day.min()).sum()
print(f'\n{leaked} training events happen AFTER the earliest test event.')
print('The model can see a user buy the item it is about to be asked to predict.')

print('\nFour leaks a clean temporal split still does not fix (rs-m5):')
for s in ['  1. features computed over the whole history (fix: as-of joins, section 7)',
          '  2. item metadata refreshed after the fact (fix: versioned snapshots)',
          '  3. the same user in train and test with no time gap (fix: a validation window)',
          '  4. popularity computed on train+test (fix: recompute inside the fold)']:
    print(s)

---
## 4 · rs-m6 / rs-b3 · Baselines

Popularity is the bar. Recency is the bar that hurts. If your model cannot beat
both, it is not a model — and you only find out by building them first.

In [ ]:
seen = train.groupby('user_id')['item_id'].apply(set).to_dict()

def rank_random(u, seed=3):
    r = np.random.default_rng(seed + int(u[1:]))
    return list(r.permutation(ITEMS))

pop = train.groupby('item_id')['user_id'].nunique().reindex(ITEMS, fill_value=0)
def rank_popularity(u):
    return list(pop.sort_values(ascending=False, kind='mergesort').index)

def rank_recency(u):
    last = train[train.user_id == u].groupby('item_id')['day'].max()
    ordered = list(last.sort_values(ascending=False, kind='mergesort').index)
    return ordered + [i for i in ITEMS if i not in ordered]

# co-visitation: items that co-occur in the same user's history
co = pd.DataFrame(0, index=ITEMS, columns=ITEMS, dtype=float)
for _, grp in train.groupby('user_id'):
    for a, b in combinations(sorted(set(grp.item_id)), 2):
        co.loc[a, b] += 1; co.loc[b, a] += 1

def rank_covisit(u):
    hist = seen.get(u, set())
    score = co.loc[list(hist)].sum() if hist else pd.Series(0.0, index=ITEMS)
    return list(score.sort_values(ascending=False, kind='mergesort').index)

print('item popularity (distinct users):')
print(pop.sort_values(ascending=False).to_string())

In [ ]:
BASELINES = {'random': rank_random, 'popularity': rank_popularity,
             'recency': rank_recency, 'co-visitation': rank_covisit}

def scoreboard(models, k=3):
    rows = []
    for nm, fn in models.items():
        m, _ = evaluate(fn, targets, k)
        rows.append(dict(model=nm, **{key: round(v, 4) for key, v in m.items()}))
    return pd.DataFrame(rows).sort_values('NDCG', ascending=False)

board = scoreboard(BASELINES)
print(board.to_string(index=False))

_, detail = evaluate(rank_recency, targets, 3)
print('\nrecency, per user (rank of the held-out target):')
print(detail[['user','target','rank','hit','rr','ndcg']].to_string(index=False))
print('\nRecency wins because e-commerce users re-visit what they just looked at.')
print('Any model that cannot beat this is losing to a two-line SQL query.')

---
## 5 · rs-m7 · Collaborative filtering

`score(u, i) = sum over the user's history j of sim(i, j) * r_uj`. The similarity
is cosine over the columns (item-item) or rows (user-user) of R.

In [ ]:
def cosine_matrix(M):
    """M: (n, d). Returns the (n, n) cosine similarity table."""
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    return (M / norms) @ (M / norms).T

Rv = R.values
item_sim = pd.DataFrame(cosine_matrix(Rv.T), index=ITEMS, columns=ITEMS)
user_sim = pd.DataFrame(cosine_matrix(Rv),   index=USER_IDS, columns=USER_IDS)

print('item-item cosine (rounded):')
print(item_sim.round(2).to_string())
print(f'\nP9 is similar to nothing: row sum {item_sim.loc["P9"].sum():.1f} '
      '— an all-zero column has no direction to compare.')
assert item_sim.loc['P9'].drop('P9').abs().sum() == 0

**Reading the next cell — the two directions.** Same matrix, transposed.

- **item-item**: "you liked P3; P3 is similar to P5, so here is P5."
  `score(u,i) = sum over the user's history j of sim(i,j) * r_uj`.
  Similarity is over **columns** of R — two items are similar if the same people
  touched them.
- **user-user**: "people like you bought P4."
  `score(u,i) = sum over other users v of sim(u,v) * r_vi`, normalised by the
  similarity weights so the result stays on the rating scale.
  Similarity is over **rows** — two users are similar if they touched the same items.

Item-item is what almost everyone ships: item vectors change slowly so the
similarity table can be precomputed nightly, whereas a user's row changes every
click and would need recomputing live.

In [ ]:
def rank_item_cf(u, exclude_seen=False):
    hist = R.loc[u]
    scores = {}
    for target in ITEMS:
        s = sum(item_sim.loc[target, j] * hist[j]
                for j in ITEMS if j != target and hist[j] > 0)
        if exclude_seen and hist[target] > 0: continue
        scores[target] = s
    return [i for i, _ in sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))]

def rank_user_cf(u, exclude_seen=False):
    sims = user_sim.loc[u].drop(u)
    scores = {}
    for j in ITEMS:
        if exclude_seen and R.loc[u, j] > 0: continue
        num = (sims * R.loc[sims.index, j]).sum()
        den = sims.abs().sum()
        scores[j] = num/den if den else 0.0
    return [i for i, _ in sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))]

models = dict(BASELINES)
models['item-item CF'] = rank_item_cf
models['user-user CF'] = rank_user_cf
print(scoreboard(models).to_string(index=False))

print('\nWhere CF structurally fails:')
print('  - a cold item (P9) has an all-zero column, so similarity is 0 to everything')
print('  - a cold user has no history to weight the sum with')
print('  - popular items dominate because they co-occur with everything')

---
## 6 · rs-m8 / rs-b4 · Implicit ALS

Objective: `sum over ALL cells c_ui (p_ui - x_u . y_i)^2 + reg(||X||^2+||Y||^2)`.
Every cell participates — the observed ones just carry more confidence. The Gram
trick is what keeps that affordable.

**Reading the next cell — the Gram trick.** The objective sums over **every**
user-item cell, including the zeros. Done literally that is
`users x items x k^2` work per sweep, which does not scale.

The trick: split the confidence into `1 + (c-1)`. The `1` part is the same for
every cell, so `sum over ALL items of y_i y_i^T` is just `Y^T Y` — computed
**once per sweep** (`YtY = Y.T @ Y`). Only the `(c-1)` correction depends on the
user, and it is non-zero *only for items they actually touched*. So each user
contributes work proportional to their own history:

```
A = YtY + Yn.T @ ((Cu - 1)[:, None] * Yn) + reg*I
    ^^^^   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    all items, once      only this user's items
```

That turns `O(users * items * k^2)` into `O(nnz * k^2)` per sweep. `nz =
np.flatnonzero(R[u])` is what picks out "only this user's items".

Note the cold-item branch: an item with no interactions has an empty `nz`, so
its normal equations are singular. It is pinned at the origin, which is the
honest answer — a zero-interaction item has no learnable position.

In [ ]:
def als_loss(R, X, Y, alpha, reg):
    P_ = (R > 0).astype(float)
    Cc_ = 1 + alpha*R
    E = P_ - X @ Y.T
    return float((Cc_ * E**2).sum() + reg*((X**2).sum() + (Y**2).sum()))

def train_als(R, k=2, alpha=1.0, reg=0.1, iters=15, seed=7):
    """Hu, Koren & Volinsky 2008. The Gram trick: YtY is computed ONCE per
    sweep over all items, and each user only adds the (c-1) correction for the
    handful of items they actually touched — so a pass is O(nnz*k^2), not
    O(users*items*k^2)."""
    rng = np.random.default_rng(seed)
    nU, nI = R.shape
    X = (rng.random((nU, k)) - 0.5) * 0.2
    Y = (rng.random((nI, k)) - 0.5) * 0.2
    losses = []
    I = np.eye(k)
    for _ in range(iters):
        YtY = Y.T @ Y                                    # <- computed once
        for u in range(nU):
            nz = np.flatnonzero(R[u])
            Cu = 1 + alpha*R[u, nz]
            Yn = Y[nz]
            A = YtY + Yn.T @ ((Cu - 1)[:, None] * Yn) + reg*I
            b = Yn.T @ Cu
            X[u] = np.linalg.solve(A, b)
        XtX = X.T @ X
        for i in range(nI):
            nz = np.flatnonzero(R[:, i])
            if len(nz) == 0:
                Y[i] = 0.0                               # cold item: stays at the origin
                continue
            Ci = 1 + alpha*R[nz, i]
            Xn = X[nz]
            A = XtX + Xn.T @ ((Ci - 1)[:, None] * Xn) + reg*I
            b = Xn.T @ Ci
            Y[i] = np.linalg.solve(A, b)
        losses.append(als_loss(R, X, Y, alpha, reg))
    return X, Y, np.array(losses)

X, Y, losses = train_als(Rv, k=2, alpha=1.0, reg=0.1, iters=15)
print('loss per iteration:', losses.round(3))
# ALS is alternating least squares: each half-step solves its side exactly,
# so the objective can never go up.
assert np.all(np.diff(losses) <= 1e-9), 'ALS loss must be monotonically non-increasing'
print(f'\nmonotone decrease confirmed: {losses[0]:.3f} -> {losses[-1]:.3f}')
print('P9 factor vector:', Y[ITEMS.index("P9")].round(6), '<- zero, as it must be')
assert np.allclose(Y[ITEMS.index('P9')], 0)

In [ ]:
def rank_als(u, exclude_seen=False):
    ui = USER_IDS.index(u)
    s = {it: float(X[ui] @ Y[j]) for j, it in enumerate(ITEMS)
         if not (exclude_seen and R.loc[u, it] > 0)}
    return [i for i, _ in sorted(s.items(), key=lambda kv: (-kv[1], kv[0]))]

models['ALS (k=2)'] = rank_als
print(scoreboard(models).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(losses, marker='o', ms=3, color=TRACK)
ax[0].set_title('ALS objective per sweep', fontsize=10); ax[0].set_xlabel('iteration')
ax[1].scatter(Y[:,0], Y[:,1], c=TRACK, s=60)
for j, it in enumerate(ITEMS):
    ax[1].annotate(it, (Y[j,0], Y[j,1]), fontsize=8, xytext=(4,3), textcoords='offset points')
ax[1].scatter(X[:,0], X[:,1], c='#dc2626', marker='^', s=60)
for i, u in enumerate(USER_IDS):
    ax[1].annotate(u, (X[i,0], X[i,1]), fontsize=8, color='#dc2626',
                   xytext=(4,3), textcoords='offset points')
ax[1].set_title('latent space: items (green) and users (red)', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Sweeping k. On a 6x9 matrix more factors is mostly more overfitting.
print(f'{"k":>3}{"final loss":>12}{"NDCG@3":>9}{"HitRate":>9}')
for k in (1, 2, 3, 4, 6):
    Xk, Yk, lk = train_als(Rv, k=k, alpha=1.0, reg=0.1, iters=20)
    def rank_k(u, Xk=Xk, Yk=Yk):
        ui = USER_IDS.index(u)
        s = {it: float(Xk[ui] @ Yk[j]) for j, it in enumerate(ITEMS)}
        return [i for i, _ in sorted(s.items(), key=lambda kv: (-kv[1], kv[0]))]
    m, _ = evaluate(rank_k, targets, 3)
    print(f'{k:>3}{lk[-1]:>12.3f}{m["NDCG"]:>9.4f}{m["HitRate"]:>9.4f}')
print('\nLoss keeps falling with k while NDCG does not — the loss is measuring fit,')
print('the metric is measuring usefulness, and only one of them is your goal.')

---
## 7 · rs-m10 / rs-b5 · Point-in-time features

Every feature must be computed **only from events strictly before** the row's
timestamp. `merge_asof` is the tool; the assertion afterwards is the proof.

In [ ]:
# Time decay: an event one half-life old counts exactly half (rs-m10).
def decay_factor(age_days, half_life):
    return 2.0 ** (-age_days / half_life)

def decayed_affinity(ev, u, as_of_day, half_life=7):
    h = ev[(ev.user_id == u) & (ev.day < as_of_day)]
    out = {i: 0.0 for i in ITEMS}
    for _, e in h.iterrows():
        out[e.item_id] += e.weight * decay_factor(as_of_day - e.day, half_life)
    return out

raw_aff = {i: 0.0 for i in ITEMS}
for _, e in train[train.user_id == 'U1'].iterrows():
    raw_aff[e.item_id] += e.weight
dec_aff = decayed_affinity(train, 'U1', as_of_day=25, half_life=7)

cmp = pd.DataFrame({'raw': raw_aff, 'decayed(hl=7)': dec_aff}).round(3)
print('U1 affinity, as of day 25:\n')
print(cmp[cmp.raw > 0].to_string())
print(f'\nhalf-life 7: an event 7 days old counts {decay_factor(7,7):.2f}, '
      f'14 days {decay_factor(14,7):.2f}, 21 days {decay_factor(21,7):.3f}')
assert abs(decay_factor(7, 7) - 0.5) < 1e-12

**Reading the next cell — `merge_asof`.** A normal join matches on equality. An
as-of join matches on "the most recent row **at or before** this timestamp",
which is exactly the question a point-in-time feature asks.

The four arguments that matter:

- `on='day'` — the time key. **Both frames must be sorted by it** or the result
  is silently wrong.
- `by='user_id'` — match within a user, so U1's features never come from U2.
- `direction='backward'` — look into the past only. This is the whole point.
- `allow_exact_matches=False` — a feature must come from **strictly before** the
  label event, not from the event itself. Leaving this at its `True` default is
  the single most common way to leak the label into its own features.

In [ ]:
# The as-of join. Sort both sides by time, then merge BACKWARD so each row sees
# only the past. This is the scalable version of "recompute the feature per row".
hist = (train.sort_values('day')
             .assign(cum_events=lambda d: d.groupby('user_id').cumcount() + 1,
                     cum_weight=lambda d: d.groupby('user_id')['weight'].cumsum()))

# label rows: one per (user, day-of-target)
labels = (pd.DataFrame([{'user_id': u, 'day': test[test.user_id == u].day.min(),
                         'item_id': t} for u, t in targets.items()])
            .sort_values('day'))

feat = pd.merge_asof(labels, hist[['day','user_id','cum_events','cum_weight']].sort_values('day'),
                     on='day', by='user_id', direction='backward', allow_exact_matches=False)
print(feat.to_string(index=False))

# THE assertion. Every feature row must be built from strictly-earlier events.
merged = feat.merge(hist[['user_id','day','cum_events']].rename(columns={'day':'src_day'}),
                    on=['user_id','cum_events'], how='left')
assert (merged.src_day < merged.day).all(), 'point-in-time violation: a feature used the future'
print('\nleak check passed: every feature value comes from a strictly earlier event')

In [ ]:
# Training/serving skew: the same feature must be computed the same way in both
# places. The usual bug is offline using the full history and online using a
# 30-day window.
def user_features(ev, u, as_of_day, window=None):
    h = ev[(ev.user_id == u) & (ev.day < as_of_day)]
    if window: h = h[h.day >= as_of_day - window]
    return dict(n_events=len(h), n_items=h.item_id.nunique(),
                total_weight=h.weight.sum(),
                days_since_last=(as_of_day - h.day.max()) if len(h) else np.nan)

full = user_features(train, 'U2', 25)
win  = user_features(train, 'U2', 25, window=10)
print('U2 as of day 25')
print('  offline (full history):', full)
print('  online  (10-day window):', win)
print('\nSame name, different number. The model learned on the first and is')
print('served the second — that is training/serving skew, and it shows up as')
print('a model that validates well and underperforms in production.')
assert full['n_events'] != win['n_events']

---
## 8 · rs-m9 / rs-b6 · Learning to rank

Positives are the held-out purchases; negatives are **sampled**, because scoring
every non-interaction would be 99% background. Then a gradient-boosted ranker
learns to order them.

**Reading the next cell — where the training labels come from.** This is the
part that is easy to get wrong, and getting it wrong flatters your metrics.

A ranker needs `(user, item, label)` rows. The positive is obvious; the trap is
**which** positive. Using the held-out test target would mean training on the
answer, so instead each user's positive is their **last event inside the
training window**, with features computed strictly before it — leave-one-out,
applied to the training period only. The test targets stay untouched until
evaluation.

Negatives are sampled rather than enumerated: with 9 items you *could* use all
of them, but at catalogue scale 99.99% of pairs are non-interactions and the
loss would be all background. Four per positive is the usual starting ratio.

`as_of_day` is carried on every row so the assertion afterwards can prove no
feature came from the future.

In [ ]:
meta = PRODUCTS.set_index('item_id')

def build_training_rows(ev, n_neg=4, seed=0, half_life=7):
    """Leave-one-out INSIDE the training window: each user's last training
    interaction is the positive, and its features are computed from events
    strictly before it. The held-out test targets are never used here — training
    on them would be a label leak, and it inflates every metric downstream."""
    rng = np.random.default_rng(seed)
    rows = []
    for u in sorted(ev.user_id.unique()):
        h = ev[ev.user_id == u].sort_values('day')
        pos_row = h.iloc[-1]
        pos, as_of = pos_row.item_id, int(pos_row.day)
        past = h[h.day < as_of]                       # strictly before the positive
        aff = decayed_affinity(ev, u, as_of, half_life)
        hist = set(past.item_id)
        cands = [pos] + list(rng.choice([i for i in ITEMS if i != pos],
                                        size=n_neg, replace=False))
        for it in cands:
            rows.append(dict(
                user_id=u, item_id=it, label=int(it == pos), as_of_day=as_of,
                affinity=aff[it],
                pop=float(pop.get(it, 0)),
                seen_before=int(it in hist),
                price=float(meta.price[it]),
                same_cat=int(meta.cat[it] in set(meta.cat[list(hist)])) if hist else 0,
            ))
    return pd.DataFrame(rows)

rows = build_training_rows(train)
print(rows.to_string(index=False))
print(f'\n{rows.label.sum()} positives, {(1-rows.label).sum()} negatives '
      f'({rows.label.mean():.0%} positive)')
assert rows.groupby('user_id').label.sum().eq(1).all(), 'exactly one positive per user'
# The positives must come from the TRAIN window, never from the held-out targets.
pos_items = rows[rows.label == 1].set_index('user_id').item_id.to_dict()
assert all(rows[rows.label == 1].as_of_day <= CUTOFF)
print('\ntrain-window positives:', pos_items)
print('held-out targets      :', targets, ' <- disjoint from the training labels for most users')

**Reading the next cell — `group`.** This is what makes it *ranking* rather than
classification.

A classifier asks "is this pair a 1 or a 0?" and scores every row on its own.
A ranker only cares about the order **within one user's candidate list** — it is
fine for U1's best item to score 0.3 while U2's best scores 0.9, because those
numbers are never compared.

`group=[5, 5, 5, 5, 5, 5]` says "the first 5 rows are one list, the next 5 are
the next list, ...". Hence the `sort_values('user_id')` immediately before:
**group sizes are positional**, so if the rows are not contiguous per user the
model silently learns to rank across the wrong boundaries.

`objective='lambdarank'` is the pairwise loss — it optimises swaps within a
group, weighted by how much each swap would move NDCG. The `try/except` falls
back to a pointwise classifier so the notebook still runs without LightGBM.

In [ ]:
FEATS = ['affinity', 'pop', 'seen_before', 'price', 'same_cat']
rows = rows.sort_values('user_id')
groups = rows.groupby('user_id', sort=True).size().values

try:
    from lightgbm import LGBMRanker
    ranker = LGBMRanker(objective='lambdarank', n_estimators=60, learning_rate=0.1,
                        min_child_samples=1, min_data_in_bin=1, num_leaves=4, verbose=-1)
    ranker.fit(rows[FEATS], rows.label, group=groups)
    importance = pd.Series(ranker.feature_importances_, index=FEATS)
    score_fn = lambda df: ranker.predict(df[FEATS])
    backend = 'LightGBM LGBMRanker (lambdarank, pairwise)'
except Exception as e:
    from sklearn.ensemble import GradientBoostingClassifier
    clf = GradientBoostingClassifier(n_estimators=60, max_depth=2).fit(rows[FEATS], rows.label)
    importance = pd.Series(clf.feature_importances_, index=FEATS)
    score_fn = lambda df: clf.predict_proba(df[FEATS])[:, 1]
    backend = f'sklearn GradientBoosting (pointwise fallback: {type(e).__name__})'

print('backend:', backend)
print('\nfeature importance:')
print(importance.sort_values(ascending=False).to_string())

In [ ]:
def rank_ltr(u):
    # Serving time: as-of the end of the training window, using the FULL train
    # history — the same feature definitions as at training, one day later.
    as_of = train[train.user_id == u].day.max() + 1
    aff = decayed_affinity(train, u, as_of)
    hist = set(train[train.user_id == u].item_id)
    cand = pd.DataFrame([dict(item_id=i, affinity=aff[i], pop=float(pop.get(i,0)),
                              seen_before=int(i in hist), price=float(meta.price[i]),
                              same_cat=int(meta.cat[i] in set(meta.cat[list(hist)])) if hist else 0)
                         for i in ITEMS])
    cand['score'] = score_fn(cand)
    return list(cand.sort_values(['score','item_id'], ascending=[False, True]).item_id)

models['LTR'] = rank_ltr
print(scoreboard(models).to_string(index=False))

print('\nRead this honestly. The ranker clears recency but not user-user CF, and on')
print('6 users with 30 training rows none of these gaps is statistically real —')
print('one user changing rank moves NDCG by ~0.17. What the pipeline buys you is')
print('the ABILITY to add a feature and measure it; the win comes from data.')
print('\nBefore the leave-one-out fix above, this table showed LTR at 0.82 NDCG,')
print('because the training positives WERE the held-out targets. That is what a')
print('label leak looks like: not an error, just an implausibly good number.')

---
## 9 · rs-m11 / rs-m12 · Negative sampling and cold start

In [ ]:
# How you pick negatives changes what the model learns.
def sample_negatives(u, n, strategy='uniform', beta=0.75, seed=0):
    rng = np.random.default_rng(seed + int(u[1:]))
    pool = [i for i in ITEMS if i != targets[u]]
    p = np.ones(len(pool))
    if strategy == 'popularity':
        p = np.array([pop.get(i, 0) + 1e-9 for i in pool]) ** beta
    elif strategy == 'hard':                     # near-misses: high model score, not the target
        s = np.array([float(X[USER_IDS.index(u)] @ Y[ITEMS.index(i)]) for i in pool])
        p = np.exp(5*(s - s.max()))
    p = p / p.sum()
    return list(rng.choice(pool, size=n, replace=False, p=p))

for strat in ('uniform', 'popularity', 'hard'):
    counts = {}
    for u in USER_IDS:
        for it in sample_negatives(u, 4, strat, seed=1):
            counts[it] = counts.get(it, 0) + 1
    top = [(str(i), c) for i, c in sorted(counts.items(), key=lambda kv: -kv[1])[:4]]
    print(f'{strat:<12} most-sampled negatives: {top}')

print('\nUniform mostly draws items nobody sees, so the model learns an easy task.')
print('popularity^beta draws items the user genuinely skipped — a real negative.')
print('Hard negatives are the most informative and the most dangerous: some of')
print('them are false negatives (items the user would have liked but never saw).')

In [ ]:
# Cold start. P9 has no interactions, so every collaborative signal is zero.
print('P9 score under each model:')
for nm, fn in models.items():
    r = fn('U1')
    print(f'  {nm:<16} rank {r.index("P9")+1} of {len(ITEMS)}')

# Content signal: P9 is in the same category as P1, which U1 actually bought.
meta = PRODUCTS.set_index('item_id')
cat_onehot = pd.get_dummies(meta.cat).astype(float)
price_n = ((meta.price - meta.price.mean()) / meta.price.std()).to_frame('price_z')
content = pd.concat([cat_onehot, price_n], axis=1)
content_sim = pd.DataFrame(cosine_matrix(content.values), index=ITEMS, columns=ITEMS)
print(f'\ncontent similarity P9 <-> P1 (both "audio"): {content_sim.loc["P9","P1"]:.3f}')
print(f'collaborative similarity P9 <-> P1:            {item_sim.loc["P9","P1"]:.3f}')
assert content_sim.loc['P9','P1'] > item_sim.loc['P9','P1']

def rank_hybrid(u, w=0.4):
    """Blend the collaborative score with a content score. The content half is
    the only thing that can rank a zero-interaction item at all."""
    ui = USER_IDS.index(u)
    hist = [i for i in ITEMS if R.loc[u, i] > 0]
    out = {}
    for j, it in enumerate(ITEMS):
        collab = float(X[ui] @ Y[j])
        cont = float(np.mean([content_sim.loc[it, h] for h in hist])) if hist else 0.0
        out[it] = (1-w)*collab + w*cont
    return [i for i, _ in sorted(out.items(), key=lambda kv: (-kv[1], kv[0]))]

models['hybrid (ALS + content)'] = rank_hybrid
print(f'\nP9 rank for U1: ALS {rank_als("U1").index("P9")+1} -> hybrid {rank_hybrid("U1").index("P9")+1}')
print('\n' + scoreboard(models).to_string(index=False))
print('\nThe hybrid scores WORSE on the warm users than plain ALS. That is the')
print('cold-start trade in one line: you give up accuracy where you had signal')
print('to buy the ability to rank an item that has none. Whether that is worth it')
print('depends on how much of your catalog is cold, which is a business fact,')
print('not a modelling one.')
print('\nCoverage is the other column to watch: a model that scores well by showing')
print('everyone the same three items has solved the metric, not the problem —')
print('compare popularity (coverage 0.33) against recency (0.89).')

---
## Where to go next

- **Scale it.** Swap the dense matrix for `scipy.sparse`, and `numpy.linalg.solve`
  for the `implicit` library — the ALS above is the same algorithm, just readable.
- **Retrieval + ranking.** This notebook ranks all 9 items. At a real catalog
  size you retrieve a few hundred candidates first (ANN over the ALS factors,
  e.g. FAISS) and only then run the ranker.
- **Sequence models.** GRU4Rec / SASRec treat the history as a sequence rather
  than a bag, which is where the RNN and Transformer tracks connect.
- **Bias.** Position bias and exposure bias both make your logs disagree with
  reality; `rs-m11` covers logQ correction, IPS weighting and randomised
  exploration.

Change `CUTOFF` from 24 and re-run: every model's score moves, and the ordering
between them is what you should actually be reading.